# Label-Agnostic SATURN Benchmark

All label-aware quantities below are computed after fine-tuning.

In [ ]:
from pathlib import Path
import json, os
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
from IPython.display import display

configured_root = os.environ.get('LABEL_AGNOSTIC_OUT')
if configured_root:
    root = Path(configured_root).expanduser().resolve()
else:
    repo_root = next((
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / 'scripts' / 'run_label_agnostic_benchmark.sh').exists()
    ), None)
    if repo_root is None:
        raise FileNotFoundError(
            'Could not locate the repository. Set LABEL_AGNOSTIC_OUT to the benchmark output directory.'
        )
    root = repo_root / 'out' / 'label_agnostic_benchmark_clean'

required_files = [root / 'comparison.csv', root / 'acceptance.json']
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Missing benchmark outputs: {missing_files}')
print(f'Loading benchmark outputs from: {root}')
assets = root / 'report_assets'
assets.mkdir(parents=True, exist_ok=True)
trials = ['baseline', 'infonce', 'mmd', 'ot']
summary = pd.read_csv(root / 'comparison.csv')
acceptance = json.loads((root / 'acceptance.json').read_text())
display(summary)
display(acceptance)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for trial in trials:
    history = pd.read_csv(root / trial / 'history.csv')
    ax.plot(history['epoch'], history['metric_loss'], marker='o', ms=3, label=trial)
ax.set(xlabel='Epoch', ylabel='Native training loss', title='Fine-tuning loss')
ax.legend()
sns.despine()
fig.tight_layout()
fig.savefig(assets / 'loss_histories.png', dpi=180)
plt.show()

In [ ]:
plot_frame = summary.melt(
    id_vars='trial',
    value_vars=['species_mixing_fraction', 'label_same_neighbor_fraction', 'normalized_species_ilisi'],
    var_name='metric', value_name='value')
fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=plot_frame, x='trial', y='value', hue='metric', ax=ax)
ax.set(title='Mixing and neighbourhood metrics', xlabel=None, ylabel='Score')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
sns.despine()
fig.tight_layout()
fig.savefig(assets / 'mixing_metrics.png', dpi=180)
plt.show()

In [ ]:
fig, axes = plt.subplots(len(trials), 2, figsize=(12, 20))
for row, trial in enumerate(trials):
    adata = sc.read_h5ad(root / trial / 'evaluated_adata.h5ad')
    sc.pl.umap(adata, color='species', ax=axes[row, 0], show=False, title=f'{trial}: species', size=8)
    sc.pl.umap(adata, color='labels2', ax=axes[row, 1], show=False, title=f'{trial}: labels', size=8, legend_loc=None)
fig.tight_layout()
fig.savefig(assets / 'trial_umaps.png', dpi=180, bbox_inches='tight')
plt.show()